In [18]:
import matplotlib
matplotlib.use('Qt5Agg')  # Important for VSCode interactivity

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import PolygonSelector
from matplotlib.path import Path
from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc

In [19]:
class PolygonAnnotator:
    def __init__(self, image, cmap='hot'):
        self.image = image
        self.coords = None
        self.done = False

        self.fig, self.ax = plt.subplots()
        self.ax.imshow(image, cmap=cmap)
        self.ax.set_title("Click to draw polygon. Double-click to finish.")

        self.selector = PolygonSelector(
            self.ax,
            self.onselect,
            useblit=True,
            props=dict(color='cyan', linewidth=2),
            handle_props=dict(marker='o', markersize=5, mec='cyan', mfc='cyan')
        )

        self.fig.canvas.mpl_connect('close_event', self._on_close)
        plt.show(block=True)  # Blocks until the window is closed

    def _on_close(self, event):
        self.done = True

    def onselect(self, verts):
        self.coords = verts
        print(f"Polygon drawn with {len(verts)} points.")
        plt.close(self.fig)  # Automatically close the window

    def get_mask(self):
        if not self.coords:
            print("No polygon was drawn.")
            return None
        ny, nx = self.image.shape
        y, x = np.mgrid[:ny, :nx]
        points = np.vstack((x.flatten(), y.flatten())).T
        path = Path(self.coords)
        return path.contains_points(points).reshape((ny, nx))

In [20]:
def get_mz_heatmap_image(adata, target_mz, tol=0.1):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    x = adata.obs["x"].values.astype(int)
    y = adata.obs["y"].values.astype(int)

    width = x.max() + 1
    height = y.max() + 1
    image = np.zeros((height, width))
    image[y, x] = intensities

    return image, matched_mz

In [4]:
# Load and visualize the m/z heatmap from AnnData
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad"
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad


In [27]:
# ---- Run ----

target_mz = 788.5
image, matched_mz = get_mz_heatmap_image(adata, target_mz)
print(f"Target m/z: {target_mz} → Matched m/z: {matched_mz}")

custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])

# Use the annotator with your image and colormap
annotator = PolygonAnnotator(image, cmap=custom_cmap)
mask = annotator.get_mask()

if mask is not None:
    masked_image = np.where(mask, image, 0)
    plt.imshow(masked_image, cmap=custom_cmap)
    plt.title(f"Masked m/z = {matched_mz:.4f}")
    plt.show()
else:
    print("No mask created.")

Target m/z: 788.5 → Matched m/z: 788.5027
Polygon drawn with 4 points.


In [ ]:
# Get spatial coordinates
x = adata.obs["x"].astype(int).values
y = adata.obs["y"].astype(int).values

# Mask value at each spot
adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)
adata.obs["tissue"] = adata.obs["tissue"].astype(int)  # 1 = inside, 0 = outside
print(adata.obs["tissue"].value_counts())

In [31]:
adata.obs["tissue"]

0        1
1        1
2        1
3        1
4        1
        ..
16600    0
16601    0
16602    0
16603    0
16604    0
Name: tissue, Length: 16605, dtype: int64